# Evaluate the raw ensemble and develop a first baseline

Keep in mind that we use the whole dataset to fit the standardscaler

## Full Dataset
The energy score of the raw ensemble with non-normalized values is [27.550657, 20.77709] for t2m and 10m wind speed.

For normalized values, the energy score is [3.3721657, 10.583179] which is 6.97767235 averaged

## Validation Dataset
For the ensemble, the loss is [3.3538413, 10.623285] which averages to  6.988565

From preliminary testing the best chen model reaches a score of 6.039 which is a bit better acutally.
I think there is lots of room for improvement.

## Training Dataset

In [1]:
%load_ext autoreload
%autoreload 2

In [21]:
from genpp.data import FORECAST_ENS_PATH, OBSERVATIONS_PATH, FC_VARS, MISSING_DAYS
from genpp.data.utils import get_time_intersection
import xarray as xr
import dask as da  # noqa: F401
from scoringrules import es_ensemble
from genpp.models.loss import EnergyScore, VariogramScore, EnsembleCRPS
from einops import rearrange
import numpy as np
from tqdm import trange, tqdm

In [ ]:
def load_data():
    ens = xr.open_dataset(
        FORECAST_ENS_PATH,
        chunks={
            "time": "auto",
            "number": -1,
            "latitude": -1,
            "longitude": -1,
            "level": -1,
        },
    )
    ens = ens[FC_VARS]

    obs = xr.open_dataset(
        OBSERVATIONS_PATH,
        chunks="auto",
    )
    obs = obs[FC_VARS]

    # Cut out the missing days first, since they are in time, not prediction_time
    ens = ens.sel(time=~ens.time.isin(MISSING_DAYS))
    ens = ens.assign_coords(prediction_time=ens.time + ens.prediction_timedelta).swap_dims(
        {"time": "prediction_time"}
    )

    times = get_time_intersection(ens, obs)

    ens = ens.sel(prediction_time=times)
    obs = obs.sel(time=times)
    ens = (
        ens.to_array()
        .rename({"variable": "feature"})
        .transpose("prediction_time", "number", "feature", "longitude", "latitude")
    )

    obs = obs.to_array().rename({"variable": "feature"})
    obs = obs.transpose("time", "feature", "longitude", "latitude")
    return ens, obs

In [5]:
ens, obs = load_data()
ens_npy = ens.values
obs_npy = obs.values

In [5]:
obs_npy: np.ndarray = rearrange(obs_npy, "t v x y -> t v (x y)")  # type: ignore
ens_npy: np.ndarray = rearrange(ens_npy, "t n v x y -> t n v (x y)")  # type: ignore

In [7]:
# Process data in chunks of 100 time steps to avoid memory issues
chunk_size = 100
results = []

total_time_steps = obs_npy.shape[0]
print(f"Processing {total_time_steps} time steps in chunks of {chunk_size}")

for i in trange(0, total_time_steps, chunk_size):
    end_idx = min(i + chunk_size, total_time_steps)

    # Extract chunk
    obs_chunk = obs_npy[i:end_idx]
    ens_chunk = ens_npy[i:end_idx]

    # Compute ensemble score for this chunk
    res_chunk = es_ensemble(obs_chunk, ens_chunk, m_axis=1, v_axis=-1)
    results.append(res_chunk)

# Concatenate all results
res = np.concatenate(results, axis=0)
print(f"Final result shape: {res.shape}")
res

Processing 3647 time steps in chunks of 100


100%|██████████| 37/37 [00:30<00:00,  1.21it/s]

Final result shape: (3647, 2)


array([[21.03015 , 21.043549],
       [25.522875, 38.481678],
       [16.232574, 29.272585],
       ...,
       [22.166098, 17.220913],
       [19.835793, 21.254383],
       [24.535027, 33.35298 ]], shape=(3647, 2), dtype=float32)

In [8]:
res.mean(axis=0)

array([27.550657, 20.77709 ], dtype=float32)

## Normalized

In [9]:
obs_mean = np.mean(obs_npy, axis=0, keepdims=True)
ens_mean = np.mean(ens_npy, axis=(0, 1), keepdims=True)

obs_std = np.std(obs_npy, axis=0, keepdims=True)
ens_std = np.std(ens_npy, axis=(0, 1), keepdims=True)

obs_standardized = (obs_npy - obs_mean) / obs_std
ens_standardized = (ens_npy - ens_mean) / ens_std

In [10]:
# Process data in chunks of 100 time steps to avoid memory issues
chunk_size = 100
results = []

total_time_steps = obs_standardized.shape[0]
print(f"Processing {total_time_steps} time steps in chunks of {chunk_size}")

for i in trange(0, total_time_steps, chunk_size):
    end_idx = min(i + chunk_size, total_time_steps)

    # Extract chunk
    obs_chunk = obs_standardized[i:end_idx]
    ens_chunk = ens_standardized[i:end_idx]

    # Compute ensemble score for this chunk
    res_chunk = es_ensemble(obs_chunk, ens_chunk, m_axis=1, v_axis=-1)
    results.append(res_chunk)

# Concatenate all results
res = np.concatenate(results, axis=0)
print(f"Final result shape: {res.shape}")
res

Processing 3647 time steps in chunks of 100


100%|██████████| 37/37 [00:32<00:00,  1.13it/s]

Final result shape: (3647, 2)


array([[ 2.3382776, 11.236837 ],
       [ 3.2511346, 19.073612 ],
       [ 1.709325 , 14.992278 ],
       ...,
       [ 2.4950917,  8.521025 ],
       [ 2.321423 ,  9.540014 ],
       [ 3.0114188, 14.1451435]], shape=(3647, 2), dtype=float32)

In [11]:
res.mean(axis=0)

array([ 3.3721657, 10.583179 ], dtype=float32)

In [7]:
# Calculate the energy score on the sliced data for the validation dataset
# Get the splits from the config file
import hydra
from genpp import BASE_DIR

with hydra.initialize_config_dir(
    config_dir=str(BASE_DIR / "configs" / "data" / "splits"), version_base=None
):
    cfg = hydra.compose(
        config_name="standard",
    )
    splits = hydra.utils.instantiate(cfg)

In [8]:
val_indices = obs.indexes["time"].get_indexer(obs.sel(time=splits.val).time)

In [14]:
res_val = res[val_indices]
res_val.mean(axis=0)

array([ 3.3538413, 10.623285 ], dtype=float32)

In [15]:
res_val.mean()

np.float32(6.988565)

# CRPS Scores for the Ensemble
| Dataset | Preprocessing | Variable | Score |
|---------|---------------|----------|-------|
| Full Dataset | Unnormalized | 2m_temperature | 0.6257 |
| Full Dataset | Unnormalized | 10m_wind_speed | 0.4701 |
| Validation Dataset | Unnormalized | 2m_temperature | 0.6221 |
| Validation Dataset | Unnormalized | 10m_wind_speed | 0.4707 |
| Validation Dataset | Normalized | 2m_temperature | 0.0795 |
| Validation Dataset | Normalized | 10m_wind_speed | 0.0334 |

**Notes:**
- Temperature uses standard scaling normalization
- Wind speed uses min-max transformation normalization -> absolute scale is smaller ~[0,1] so scores are even smaller
- Scores represent CRPS (Continuous Ranked Probability Score) values
- Lower scores indicate better performance


In [25]:
from genpp.models.loss import CRPS_Normal, CRPS_TruncatedNormal
import torch

In [23]:
ens, obs = load_data()
ens_mean = ens.mean(dim="number", keep_attrs=True)
ens_std = ens.std(dim="number", keep_attrs=True)

In [24]:
scoring_fns = [CRPS_Normal(), CRPS_TruncatedNormal()]
for feat, score in zip(FC_VARS, scoring_fns):
    feat_mean = torch.Tensor(ens_mean.sel(feature=feat).values)
    feat_std = torch.Tensor(ens_std.sel(feature=feat).values)
    obs_torch = torch.Tensor(obs.sel(feature=feat).values)
    s = score(feat_mean, feat_std, obs_torch)
    print(f"Score for {feat}: {s.mean()}")

Score for 2m_temperature: 0.6257416009902954
Score for 10m_wind_speed: 0.47011083364486694


## Val Dataset

In [ ]:
scoring_fns = [CRPS_Normal(), CRPS_TruncatedNormal()]
for feat, score in zip(FC_VARS, scoring_fns):
    feat_mean = torch.Tensor(ens_mean.sel(feature=feat).values)[val_indices]
    feat_std = torch.Tensor(ens_std.sel(feature=feat).values)[val_indices]
    obs_torch = torch.Tensor(obs.sel(feature=feat).values)[val_indices]
    s = score(feat_mean, feat_std, obs_torch)
    print(f"Score for {feat}: {s.mean()}")

In [23]:
es = EnergyScore()
vs = VariogramScore(p=0.5)
crps = EnsembleCRPS()

In [15]:
obs_val = obs_npy[val_indices]
ens_val = ens_npy[val_indices]

In [27]:
obs_val_per_var = rearrange(obs_val, "t d lon lat -> t d (lon lat)")
ens_val_per_var = rearrange(ens_val, "t n d lon lat -> t d n (lon lat)")

obs_val_complete = rearrange(obs_npy, "t d lon lat -> t (d lon lat)")
ens_val_complete = rearrange(ens_npy, "t n d lon lat -> t n (d lon lat)")

es_per_var = es(torch.Tensor(ens_val_per_var), torch.Tensor(obs_val_per_var))
es_complete = es(torch.Tensor(ens_val_complete), torch.Tensor(obs_val_complete))

crps_per_margin = crps(torch.Tensor(ens_val), torch.Tensor(obs_val))

vss = []
for x_i, y_i in tqdm(zip(ens_val_per_var, obs_val_per_var), total=ens_val_per_var.shape[0]):
    vss.append(vs(torch.Tensor(x_i), torch.Tensor(y_i)))
variogram_score_per_var = torch.stack(vss)


vss = []
for x_i, y_i in tqdm(zip(ens_val_complete, obs_val_complete), total=ens_val_complete.shape[0]):
    vss.append(vs(torch.Tensor(x_i), torch.Tensor(y_i)))
variogram_score_full = torch.stack(vss)

100%|██████████| 3647/3647 [21:21<00:00,  2.85it/s]


In [ ]:
from einops import reduce

print(f"Mean es for t2m and 10m wind speed: {es_per_var.mean(dim=0)}")
print(f"Mean vs for t2m and 10m wind speed: {variogram_score_per_var.mean(dim=0)}")
print(
    f"Mean crps for t2m and 10m wind speed: {reduce(crps_per_margin, 't f lat lon -> f', 'mean')}"
)
print(f"Mean es complete: {es_complete.mean(dim=0)}")
print(f"Mean vs complete: {variogram_score_full.mean(dim=0)}")
print(f"Mean crps complete: {reduce(crps_per_margin, 't f lat lon -> 1', 'mean').item()}")

Mean es for t2m and 10m wind speed: tensor([27.3330, 20.8668])
Mean vs for t2m and 10m wind speed: tensor([236289.8594, 197903.6875])
Mean crps for t2m and 10m wind speed: tensor([0.6252, 0.4737])
Mean es complete: 34.925132751464844
Mean vs complete: 436576.25
Mean crps complete: 0.5494358539581299


## Normalized

In [26]:
from genpp.preproc.preprocessors import StandardScalerPreprocessor, MinMaxScalerPreprocessor

In [ ]:
standardize = StandardScalerPreprocessor(
    dim=["prediction_time", "number", "latitude", "longitude"], features=["2m_temperature"]
)
minmax = MinMaxScalerPreprocessor(
    dim=["prediction_time", "number", "latitude", "longitude"], features=["10m_wind_speed"]
)
standardize.fit(ens)
minmax.fit(ens)

standardize_obs = StandardScalerPreprocessor(
    dim=["time", "longitude", "latitude"], features=["2m_temperature"]
)
minmax_obs = MinMaxScalerPreprocessor(
    dim=["time", "longitude", "latitude"], features=["10m_wind_speed"]
)
standardize_obs.fit(obs)
minmax_obs.fit(obs)

In [47]:
ens_transformed = standardize.preprocess(ens)
ens_mean_transformed = minmax.preprocess(ens_transformed)

obs_transformed = standardize_obs.preprocess(obs)
obs_transformed = minmax_obs.preprocess(obs_transformed)

In [49]:
ens_transformed_mean = ens_transformed.mean(dim="number")
ens_transformed_std = ens_transformed.std(dim="number")

In [50]:
scoring_fns = [CRPS_Normal(), CRPS_TruncatedNormal()]
for feat, score in zip(FC_VARS, scoring_fns):
    feat_mean = torch.Tensor(ens_transformed_mean.sel(feature=feat).values)[val_indices]
    feat_std = torch.Tensor(ens_transformed_std.sel(feature=feat).values)[val_indices]
    obs_torch = torch.Tensor(obs_transformed.sel(feature=feat).values)[val_indices]
    s = score(feat_mean, feat_std, obs_torch)
    print(f"Score for {feat}: {s.mean()}")

Score for 2m_temperature: 0.07948469370603561
Score for 10m_wind_speed: 0.03335368633270264
